# Assignment 1 



Put this notebook in the same folder as:

- `support.py`
- `dataset_a1.txt`


In [ ]:
import random
"""
import time

import numpy as np
import matplotlib.pyplot as plt

from deap import base, creator, tools, algorithms

import support
"""

## GA parameters

These are the initial parameter values for Task 1. 

In [ ]:
POPULATION_SIZE = 50
N_GENERATIONS = 20

CROSSOVER_PROBABILITY = 0.7
MUTATION_PROBABILITY = 0.3

TOURNAMENT_SIZE = 3
PER_GENE_MUTATION_PROBABILITY = 0.1

RANDOM_SEED = 42

## Individual representation

A candidate Petri net is represented as a linearized matrix.

There are 12 transitions. Each transition has two numbers: start place and end place. Therefore, one individual has `12 * 2 = 24` integer values. Each integer is a place number from `0` to `8`.

In [ ]:
INDIVIDUAL_LENGTH = support.NR_TRANSITIONS * 2

LOWER_BOUND = 0
UPPER_BOUND = support.NR_PLACES - 1

print("Individual length:", INDIVIDUAL_LENGTH)
print("Allowed place values:", LOWER_BOUND, "to", UPPER_BOUND)

## DEAP setup

In [ ]:
# These checks prevent errors if this cell is run more than once.
if "FitnessMax_Task1" not in creator.__dict__:
    creator.create("FitnessMax_Task1", base.Fitness, weights=(1.0,))

if "Individual_Task1" not in creator.__dict__:
    creator.create("Individual_Task1", list, fitness=creator.FitnessMax_Task1)

toolbox = base.Toolbox()

# One gene is one place number from 0 to 8.
toolbox.register("attr_place", random.randint, LOWER_BOUND, UPPER_BOUND)

# One individual is a list of 24 place numbers.
toolbox.register(
    "individual",
    tools.initRepeat,
    creator.Individual_Task1,
    toolbox.attr_place,
    n=INDIVIDUAL_LENGTH,
)

# A population is a list of individuals.
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# Fitness function from support.py.
toolbox.register("evaluate", support.fitness)

# Required operators for Task 1.
toolbox.register("mate", tools.cxTwoPoint)

toolbox.register(
    "mutate",
    tools.mutUniformInt,
    low=LOWER_BOUND,
    up=UPPER_BOUND,
    indpb=PER_GENE_MUTATION_PROBABILITY,
)

toolbox.register(
    "select",
    tools.selTournament,
    tournsize=TOURNAMENT_SIZE,
)

## Run the GA

In [ ]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

population = toolbox.population(n=POPULATION_SIZE)
hall_of_fame = tools.HallOfFame(1)

stats = tools.Statistics(lambda individual: individual.fitness.values[0])
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)

start_time = time.time()

population, logbook = algorithms.eaSimple(
    population,
    toolbox,
    cxpb=CROSSOVER_PROBABILITY,
    mutpb=MUTATION_PROBABILITY,
    ngen=N_GENERATIONS,
    stats=stats,
    halloffame=hall_of_fame,
    verbose=True,
)

running_time = time.time() - start_time

## Results

In [ ]:
best_individual = hall_of_fame[0]
best_fitness = best_individual.fitness.values[0]

print("Parameter values used:")
print("Population size:", POPULATION_SIZE)
print("Number of generations:", N_GENERATIONS)
print("Crossover probability:", CROSSOVER_PROBABILITY)
print("Mutation probability:", MUTATION_PROBABILITY)
print("Tournament size:", TOURNAMENT_SIZE)
print("Per-gene mutation probability:", PER_GENE_MUTATION_PROBABILITY)
print("Random seed:", RANDOM_SEED)

print("\nRunning time:", round(running_time, 2), "seconds")
print("Best fitness:", best_fitness)

print("\nBest individual:")
print(list(best_individual))

print("\nBest individual as matrix:")
print(support.list_to_array(best_individual))

## Plot best fitness at each generation

In [ ]:
generations = logbook.select("gen")
best_fitness_per_generation = logbook.select("max")

plt.figure(figsize=(8, 5))
plt.plot(generations, best_fitness_per_generation, marker="o")
plt.xlabel("Generation")
plt.ylabel("Best fitness")
plt.title("Task 1: Best fitness at each generation")
plt.grid(True)
plt.tight_layout()
plt.savefig("task1_best_fitness.png", dpi=200)
plt.show()